# Sensitivity Analysis – Threshold Selection (Oulu)

This notebook evaluates how sensitive the key output metrics are to the choice of the 0.5 cumulative-exposure threshold used to define the *typical* trip in the tour and single-trip CO₂ models. The analysis covers all three modes (PT, Car, Bike) for Oulu. For each threshold value in {0.0, 0.1, …, 1.0} the following metrics are computed:

- Mean and median of `decent_mobility_co2` (g CO₂ / week per user)
- % of users below the 2030 budget (7 000 g/week)
- % of users below the 2050 budget (3 000 g/week)

A vertical line at 0.5 marks the value used in the main analysis.

## Setup

In [ ]:
### Geolibraries
import geopandas as gpd
from shapely.geometry import Polygon

# General tools
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pyarrow.parquet as pq
from h3 import h3

BUDGET_2030 = 7000   # g CO2/week
BUDGET_2050 = 3000   # g CO2/week
THRESHOLDS  = [round(t / 10, 1) for t in range(0, 11)]  # 0.0 … 1.0


## Shared data (users, POIs, jobs)

In [ ]:
users = pd.read_parquet("scratch/data/users_and_stays_3months_oulu.parquet")
users = users[users["home_gid9"].notna()]


In [ ]:
jobs = gpd.read_parquet("data/job_distribution_from_census_oulu.parquet").reset_index()
pois = pd.read_parquet("data/pois_per_hex_new_class_oulu.parquet")

users = users.merge(
    jobs[["ID", "weighted_tyo"]],
    left_on="stay_gid9", right_on="ID", how="left"
).drop(columns="ID")

pois_grouped = (
    pois.groupby(["h3_id", "category"])["count"]
    .sum().unstack(fill_value=0).reset_index()
)

users = users.merge(
    pois_grouped[[
        "h3_id", "Education", "Healthcare and Health",
        "Others / Not sure", "Recreational, Outdoors",
        "Shopping, Errands", "Social, Cultural",
    ]],
    left_on="stay_gid9", right_on="h3_id", how="left"
).drop(columns="h3_id")


In [ ]:
# Postal code lookup for user home locations
postal = gpd.read_file("./data/postal_code_finland/pno_tilasto_2024.shp")

def h3_to_polygon(h3_id):
    return Polygon(h3.h3_to_geo_boundary(h3_id, geo_json=True))

users["geometry"] = users["home_gid9"].apply(h3_to_polygon)
gdf = gpd.GeoDataFrame(users, geometry="geometry", crs="EPSG:4326").to_crs(postal.crs)

intersections = gpd.overlay(gdf, postal, how="intersection")
intersections["intersect_area"] = intersections.geometry.area
idx = intersections.groupby("home_gid9")["intersect_area"].idxmax()
largest_overlap = intersections.loc[idx]

gdf = gdf.merge(
    largest_overlap[["home_gid9", "postinumer", "nimi", "geometry"]],
    on="home_gid9", how="left"
)


## Helper: build `tour_df` with `cum_exposure_share` for a given CO₂ matrix

In [ ]:
def build_tour_df(users, df_co2, co2_col, gdf):
    """
    Replicates the tour_df construction from the main notebook.
    co2_col : column name in df_co2 used as the per-leg CO2 value (outbound).
    Returns tour_df with cum_exposure_share already computed.
    """
    activity_cols = ["Social, Cultural", "Shopping, Errands", "Recreational, Outdoors"]

    df_base = users[(users["is_home"] == 0) & (users["is_work"] == 0)].copy()

    df_long = df_base.melt(
        id_vars=["user_id", "stay_gid9", "home_gid9", "work_gid9", "frequency_period"],
        value_vars=activity_cols,
        var_name="activity_type", value_name="poi_count"
    )
    df_long = df_long[df_long["poi_count"] > 0].copy()
    df_long = df_long.rename(columns={"stay_gid9": "activity_gid9"})
    df_long["tour_id"] = df_long.groupby(["user_id", "activity_type"]).cumcount()

    def build_legs(row):
        return [
            (row.user_id, row.home_gid9,     row.work_gid9,     row.activity_type, row.frequency_period, row.poi_count),
            (row.user_id, row.work_gid9,     row.activity_gid9, row.activity_type, row.frequency_period, row.poi_count),
            (row.user_id, row.activity_gid9, row.home_gid9,     row.activity_type, row.frequency_period, row.poi_count),
        ]

    legs = df_long.apply(build_legs, axis=1)
    legs_df = pd.DataFrame(
        [leg for tour in legs for leg in tour],
        columns=["user_id", "from_id", "to_id", "tour_type", "frequency", "poi_count"]
    )

    legs_df = legs_df.merge(df_co2, on=["from_id", "to_id"], how="left")

    legs_df["tour_id"] = legs_df.index // 3

    tour_validity = (
        legs_df.groupby("tour_id")[co2_col].count()
        .reset_index(name="n_legs")
    )
    valid_tours = tour_validity[tour_validity["n_legs"] == 3]
    legs_df = legs_df.merge(valid_tours[["tour_id"]], on="tour_id", how="inner")

    tour_df = (
        legs_df
        .groupby(["user_id", "tour_id", "tour_type", "frequency", "poi_count"])[co2_col]
        .sum().reset_index()
        .rename(columns={co2_col: "tour_co2"})
    )

    tour_df["exposure"] = tour_df["frequency"] * tour_df["poi_count"]

    tour_counts = tour_df.groupby(["user_id", "tour_type"]).size().reset_index(name="n_tours")
    valid_pairs = tour_counts[tour_counts["n_tours"] >= 3]
    tour_df = tour_df.merge(valid_pairs[["user_id", "tour_type"]], on=["user_id", "tour_type"], how="inner")

    tour_df = tour_df.sort_values(["user_id", "tour_type", "tour_co2"])
    tour_df["cum_exposure"]  = tour_df.groupby(["user_id", "tour_type"])["exposure"].cumsum()
    tour_df["total_exposure"] = tour_df.groupby(["user_id", "tour_type"])["exposure"].transform("sum")
    tour_df["cum_exposure_share"] = tour_df["cum_exposure"] / tour_df["total_exposure"]

    # Merge postal info
    tour_df = tour_df.merge(
        gdf[["user_id", "home_gid9", "postinumer", "nimi"]].drop_duplicates(),
        on="user_id", how="left"
    )

    return tour_df


## Helper: run sensitivity loop for one mode

In [ ]:
def run_sensitivity(tour_df, df_typical_single, thresholds, jobs_co2_col):
    """
    For each threshold, pick the typical tour and typical single trip,
    build weekly_routine, and record summary metrics.
    df_typical_single must have columns: typical_trip_co2, typical_trip_co2_0,
    typical_trip_co2_10, ..., typical_trip_co2_100
    jobs_co2_col: ignored (jobs always fixed), included for clarity.
    """
    records = []

    for t in thresholds:
        t_int = int(round(t * 100))
        single_col = "typical_trip_co2" if t == 0.5 else f"typical_trip_co2_{t_int}"

        # --- Tour-level typical trip ---
        if t == 0.0:
            # minimum: first row per (user_id, tour_type) when sorted by tour_co2
            df_typ = (
                tour_df
                .sort_values(["user_id", "tour_type", "tour_co2"])
                .groupby(["user_id", "tour_type", "nimi", "postinumer"], as_index=False)
                .first()
                .rename(columns={"tour_co2": "typical_trip_co2"})
            )
        else:
            df_typ = (
                tour_df
                .sort_values(["user_id", "tour_type", "cum_exposure_share"])
                .loc[tour_df["cum_exposure_share"] >= t]
                .groupby(["user_id", "tour_type", "nimi", "postinumer"], as_index=False)
                .first()
                .rename(columns={"tour_co2": "typical_trip_co2"})
            )

        # --- Single-trip typical (from enriched parquet) ---
        # Rename the threshold column to typical_trip_co2_single for merging
        if single_col not in df_typical_single.columns:
            # fallback to 0.5 if column missing
            single_col = "typical_trip_co2"

        df_single_t = df_typical_single[["user_id", "poi_type", "nimi", "postinumer", single_col]].copy()
        df_single_t = df_single_t.rename(columns={single_col: "typical_trip_co2_single"})

        # --- Merge tour + single ---
        df_final = df_typ.merge(
            df_single_t,
            left_on=["user_id", "tour_type", "nimi", "postinumer"],
            right_on=["user_id", "poi_type", "nimi", "postinumer"],
            how="left", suffixes=("", "_single")
        ).drop(columns=["poi_type"], errors="ignore")

        # Jobs rows
        df_jobs = (
            df_typical_single[df_typical_single["poi_type"] == "jobs"]
            .copy()
            .rename(columns={"typical_trip_co2": "co2_jobs", "poi_type": "tour_type"})
        )
        df_jobs["tour_type"] = "jobs"
        df_jobs["tour_id"] = 0
        df_jobs["frequency"] = df_jobs["poi_count"] = 1
        df_jobs["exposure"] = df_jobs["cum_exposure"] = df_jobs["total_exposure"] = df_jobs["cum_exposure_share"] = 1
        df_jobs = df_jobs.rename(columns={"co2_jobs": "typical_trip_co2"})
        df_jobs["typical_trip_co2_single"] = df_jobs["typical_trip_co2"]
        for col in df_final.columns:
            if col not in df_jobs.columns:
                df_jobs[col] = 1
        df_jobs = df_jobs[df_final.columns]

        df_final = pd.concat([df_final, df_jobs], ignore_index=True)

        # Valid users (all 4 categories present)
        required = {"jobs", "Social, Cultural", "Shopping, Errands", "Recreational, Outdoors"}
        valid_users = (
            df_final.groupby("user_id")["tour_type"]
            .apply(lambda x: required.issubset(set(x)))
        )
        valid_users = valid_users[valid_users].index
        df = df_final[df_final["user_id"].isin(valid_users)].copy()

        # Pivot
        tour   = df.pivot(index="user_id", columns="tour_type", values="typical_trip_co2")
        single = df.pivot(index="user_id", columns="tour_type", values="typical_trip_co2_single")

        weekly = pd.DataFrame(index=tour.index)
        weekly["social_component"]     = 2 * tour["Social, Cultural"]
        weekly["recreation_component"] = 2 * tour["Recreational, Outdoors"]
        weekly["shopping_component"]   = single["Shopping, Errands"]
        weekly["decent_mobility_co2"]  = (
            weekly["social_component"] +
            weekly["recreation_component"] +
            weekly["shopping_component"]
        )
        co2 = weekly["decent_mobility_co2"].dropna()

        records.append({
            "threshold":      t,
            "n_users":        len(co2),
            "mean_co2":       co2.mean(),
            "median_co2":     co2.median(),
            "pct_below_2030": (co2 <= BUDGET_2030).mean() * 100,
            "pct_below_2050": (co2 <= BUDGET_2050).mean() * 100,
        })
        print(f"  t={t:.1f}  n={len(co2):,}  mean={co2.mean():.0f}  pct<2030={records[-1]['pct_below_2030']:.1f}%")

    return pd.DataFrame(records)


In [ ]:
def plot_sensitivity(results, mode_label):
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle(f"Sensitivity to threshold – {mode_label}", fontsize=13)

    metrics = [
        ("mean_co2",       "Mean CO₂ (g/week)",          "steelblue"),
        ("median_co2",     "Median CO₂ (g/week)",         "seagreen"),
        ("pct_below_2030", "% users below 2030 budget",   "darkorange"),
        ("pct_below_2050", "% users below 2050 budget",   "royalblue"),
    ]

    for ax, (col, ylabel, color) in zip(axes, metrics):
        ax.plot(results["threshold"], results[col], marker="o", color=color, linewidth=2)
        ax.axvline(0.5, color="red", linestyle="--", linewidth=1.2, label="0.5 (main analysis)")
        ax.set_xlabel("Threshold")
        ax.set_ylabel(ylabel)
        ax.set_xticks(THRESHOLDS)
        ax.grid(True, linestyle=":", alpha=0.5)
        ax.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(f"./output/sensitivity_{mode_label.lower()}_oulu.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: ./output/sensitivity_{mode_label.lower()}_oulu.png")


---
## PT

In [ ]:
cols_needed = ["from_id", "to_id", "pt_co2_total"]
table_pt = pq.read_table("scratch/pt_co2_3000_oulu.parquet", columns=cols_needed, use_threads=True)
df_pt_co2 = table_pt.to_pandas(types_mapper=pd.ArrowDtype)

df_pt_sym = df_pt_co2.merge(
    df_pt_co2, left_on=["from_id","to_id"], right_on=["to_id","from_id"],
    how="left", suffixes=("_outbound","_inbound")
)
df_pt_sym["pt_co2_inbound"] = df_pt_sym["pt_co2_total_inbound"].fillna(df_pt_sym["pt_co2_total_outbound"])
df_pt_sym["pt_co2_total"]   = df_pt_sym["pt_co2_total_outbound"] + df_pt_sym["pt_co2_inbound"]

df_pt_co2 = df_pt_sym.rename(columns={"from_id_outbound":"from_id","to_id_outbound":"to_id"})[
    ["from_id","to_id","pt_co2_total","pt_co2_total_outbound"]
]


In [ ]:
tour_df_pt = build_tour_df(users, df_pt_co2, "pt_co2_total_outbound", gdf)
print("PT tour_df shape:", tour_df_pt.shape)


In [ ]:
df_typical_single_pt = pd.read_parquet("scratch/pt_typ_cat_oulu.parquet")
print("PT single parquet columns:", df_typical_single_pt.columns.tolist())


In [ ]:
print("Running PT sensitivity...")
results_pt = run_sensitivity(tour_df_pt, df_typical_single_pt, THRESHOLDS, "pt_co2_total")



In [ ]:
plot_sensitivity(results_pt, "PT")

---
## Car

In [ ]:
cols_needed = ["Origin_Hexagon_ID", "Destination_Hexagon_ID", "car_co2"]
table_car = pq.read_table("scratch/car_co2_6000_oulu.parquet", columns=cols_needed, use_threads=True)
df_car_co2 = table_car.to_pandas(types_mapper=pd.ArrowDtype)
df_car_co2 = df_car_co2.rename(columns={"Origin_Hexagon_ID":"from_id","Destination_Hexagon_ID":"to_id","car_co2":"car_co2_outbound"})

inbound = df_car_co2.rename(columns={"from_id":"to_id","to_id":"from_id","car_co2_outbound":"car_co2_inbound"})
df_car_sym = df_car_co2.merge(inbound, on=["from_id","to_id"], how="left")
df_car_sym["car_co2_inbound"] = df_car_sym["car_co2_inbound"].fillna(df_car_sym["car_co2_outbound"])
df_car_sym["car_co2_total"]   = df_car_sym["car_co2_outbound"] + df_car_sym["car_co2_inbound"]

df_car_co2 = df_car_sym[["from_id","to_id","car_co2_outbound","car_co2_inbound","car_co2_total"]]


In [ ]:
tour_df_car = build_tour_df(users, df_car_co2, "car_co2_outbound", gdf)
print("Car tour_df shape:", tour_df_car.shape)


In [ ]:
df_typical_single_car = pd.read_parquet("scratch/car_typ_cat_oulu.parquet")
print("Car single parquet columns:", df_typical_single_car.columns.tolist())


In [ ]:
df_typical_single_car = df_typical_single_car.rename(columns={"Nimi": "nimi", "Posnro": "postinumer"})

In [ ]:
print("Running Car sensitivity...")
results_car = run_sensitivity(tour_df_car, df_typical_single_car, THRESHOLDS, "car_co2_outbound")
results_car


In [ ]:
plot_sensitivity(results_car, "Car")

---
## Bike

In [ ]:
cols_needed = ["from_id", "to_id", "co2_emissions_g"]
table_bike = pq.read_table("scratch/bike_co2_3000_oulu.parquet", columns=cols_needed, use_threads=True)
df_bike_co2 = table_bike.to_pandas(types_mapper=pd.ArrowDtype)
df_bike_co2 = df_bike_co2.rename(columns={"co2_emissions_g":"bike_co2_outbound"})

inbound = df_bike_co2.rename(columns={"from_id":"to_id","to_id":"from_id","bike_co2_outbound":"bike_co2_inbound"})
df_bike_sym = df_bike_co2.merge(inbound, on=["from_id","to_id"], how="left")
df_bike_sym["bike_co2_inbound"] = df_bike_sym["bike_co2_inbound"].fillna(df_bike_sym["bike_co2_outbound"])
df_bike_sym["bike_co2_total"]   = df_bike_sym["bike_co2_outbound"] + df_bike_sym["bike_co2_inbound"]

df_bike_co2 = df_bike_sym[["from_id","to_id","bike_co2_outbound","bike_co2_inbound","bike_co2_total"]]


In [ ]:
tour_df_bike = build_tour_df(users, df_bike_co2, "bike_co2_outbound", gdf)
print("Bike tour_df shape:", tour_df_bike.shape)


In [ ]:
df_typical_single_bike = pd.read_parquet("scratch/bike_typ_cat_oulu.parquet")
print("Bike single parquet columns:", df_typical_single_bike.columns.tolist())


In [ ]:
df_typical_single_bike = df_typical_single_bike.rename(columns={"Nimi": "nimi", "Posnro": "postinumer"})

In [ ]:
print("Running Bike sensitivity...")
results_bike = run_sensitivity(tour_df_bike, df_typical_single_bike, THRESHOLDS, "bike_co2_outbound")
results_bike


In [ ]:
plot_sensitivity(results_bike, "Bike")

---
## Combined summary across modes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Sensitivity to threshold – All modes (Oulu)", fontsize=14)
modes = [
    ("PT",   results_pt,   "steelblue"),
    ("Car",  results_car,  "tomato"),
    ("Bike", results_bike, "seagreen"),
]
metrics = [
    ("median_co2",     "Median CO₂ (g/week)"),
    ("pct_below_2030", "% users below 2030 budget (7 CO2 kg/week)"),
]
for ax, (col, ylabel) in zip(axes.flat, metrics):
    for label, res, color in modes:
        ax.plot(res["threshold"], res[col], marker="o", label=label, color=color, linewidth=2)
    ax.axvline(0.5, color="red", linestyle="--", linewidth=1.2, label="0.5")
    ax.set_xlabel("Threshold")
    ax.set_ylabel(ylabel)
    ax.set_xticks(THRESHOLDS)
    ax.grid(True, linestyle=":", alpha=0.5)
    ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("./output/sensitivity_all_modes_oulu.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: ./output/sensitivity_all_modes_oulu.png")

In [ ]:
# Export summary tables
results_pt["mode"]   = "PT"
results_car["mode"]  = "Car"
results_bike["mode"] = "Bike"

summary = pd.concat([results_pt, results_car, results_bike], ignore_index=True)
summary.to_csv("./output/sensitivity_summary_oulu.csv", index=False)

